# Machine Learning – Practical 3
## Linear, Ridge & Lasso Regression on the California Housing Dataset

**Aim:** To predict median house value using Linear, Ridge and Lasso Regression and compare their performance.

**Objective:**
- Load and explore the California Housing dataset
- Check multicollinearity using VIF
- Train Linear, Ridge and Lasso models (with hyperparameter tuning)
- Compare models using MSE, RMSE, MAE and R²
- Analyse coefficients and feature selection

**Theory:** Regression models a continuous target as a function of input features. Linear Regression minimises the sum of squared errors. Ridge (L2) and Lasso (L1) add a penalty on coefficient size to reduce overfitting, controlled by the hyperparameter `alpha`.

## 1. Import Libraries

**Aim:** Import all libraries required for the practical.

**Objective:** Load tools for data handling, modelling, evaluation and plotting.

**Theory:** NumPy/Pandas handle data, Matplotlib plots results, scikit-learn provides models, pipelines and metrics, and statsmodels provides the Variance Inflation Factor (VIF).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
from statsmodels.stats.outliers_influence import variance_inflation_factor

print("All libraries imported successfully!")

## 2. Load the Dataset

**Aim:** Load the California Housing dataset.

**Objective:** Obtain the feature matrix `X` and target `y` (median house value) and inspect them.

**Theory:** The dataset has 20,640 samples and 8 numerical features (e.g. median income, house age, latitude, longitude). The target is the median house value in units of $100,000.

In [ ]:
housing = fetch_california_housing(as_frame=True)

X = housing.data.copy()
y = housing.target.copy()
y.name = "MedianHouseValue"

print("Dataset Shape:", X.shape)
print("\nFeatures:")
print(list(X.columns))
print("\nFirst Five Records:")
print(X.head())
print("\nTarget Variable:")
print(y.head())

## 3. Train-Test Split & Multicollinearity Check (VIF)

**Aim:** Split the data and detect multicollinearity among features.

**Objective:** Create an 80/20 train-test split and compute VIF on the scaled training features.

**Theory:** The train-test split evaluates generalisation on unseen data. The Variance Inflation Factor measures how much a feature is explained by the other features: VIF ≈ 1 means no correlation, VIF > 5 suggests notable multicollinearity, and VIF > 10 is severe. Regularisation (Ridge/Lasso) helps when multicollinearity exists.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

vif_scaler = StandardScaler()
X_train_scaled = vif_scaler.fit_transform(X_train)

vif_data = pd.DataFrame()
vif_data["Feature"] = X_train.columns
vif_data["VIF"] = [
    variance_inflation_factor(X_train_scaled, i)
    for i in range(X_train_scaled.shape[1])
]

print("Variance Inflation Factor (VIF):")
print(vif_data.sort_values("VIF", ascending=False).round(3))

## 4. Evaluation Function

**Aim:** Create a reusable function to evaluate regression models.

**Objective:** Compute MSE, RMSE, MAE and R² for both train and test sets.

**Theory:**
- **MSE** – average squared error (penalises large errors)
- **RMSE** – square root of MSE, in target units
- **MAE** – average absolute error
- **R²** – proportion of variance explained by the model (closer to 1 is better)

In [ ]:
def evaluate_model(model_name, y_train, train_pred, y_test, test_pred):
    return {
        "Model": model_name,
        "Train MSE": mean_squared_error(y_train, train_pred),
        "Test MSE": mean_squared_error(y_test, test_pred),
        "Train RMSE": np.sqrt(mean_squared_error(y_train, train_pred)),
        "Test RMSE": np.sqrt(mean_squared_error(y_test, test_pred)),
        "Train MAE": mean_absolute_error(y_train, train_pred),
        "Test MAE": mean_absolute_error(y_test, test_pred),
        "Train R2": r2_score(y_train, train_pred),
        "Test R2": r2_score(y_test, test_pred)
    }

results = []
print("Evaluation function created successfully!")

## 5. Linear Regression

**Aim:** Build a baseline Linear Regression model.

**Objective:** Fit an ordinary least squares model on standardised features and record its metrics.

**Theory:** Linear Regression fits ŷ = w₀ + w₁x₁ + … + wₙxₙ by minimising the sum of squared residuals. It has no regularisation, so it can be unstable when features are correlated. A `Pipeline` with `StandardScaler` ensures scaling is learned only from training data.

In [ ]:
linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

linear_train_pred = linear_model.predict(X_train)
linear_test_pred = linear_model.predict(X_test)

results.append(
    evaluate_model("Linear Regression", y_train, linear_train_pred, y_test, linear_test_pred)
)

print("Linear Regression completed successfully!")

## 6. Ridge Regression

**Aim:** Apply Ridge (L2) Regression with tuned regularisation strength.

**Objective:** Use 5-fold `GridSearchCV` to find the best `alpha` and evaluate the tuned model.

**Theory:** Ridge minimises  Σ(y − ŷ)² + α·Σw². The L2 penalty shrinks coefficients towards zero (but never exactly to zero), reducing variance and handling multicollinearity. A larger `alpha` means stronger shrinkage.

In [ ]:
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

ridge_params = {"model__alpha": np.logspace(-4, 4, 50)}

ridge_grid = GridSearchCV(
    ridge_pipeline,
    ridge_params,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

ridge_grid.fit(X_train, y_train)

best_ridge = ridge_grid.best_estimator_
ridge_train_pred = best_ridge.predict(X_train)
ridge_test_pred = best_ridge.predict(X_test)

results.append(
    evaluate_model("Ridge Regression", y_train, ridge_train_pred, y_test, ridge_test_pred)
)

print("Best Ridge Parameter:")
print(ridge_grid.best_params_)
print("Ridge Regression completed successfully!")

## 7. Lasso Regression

**Aim:** Apply Lasso (L1) Regression with tuned regularisation strength.

**Objective:** Use 5-fold `GridSearchCV` to find the best `alpha` and evaluate the tuned model.

**Theory:** Lasso minimises  Σ(y − ŷ)² + α·Σ|w|. The L1 penalty can shrink some coefficients exactly to zero, performing automatic feature selection and producing simpler, more interpretable models.

In [ ]:
lasso_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(max_iter=10000))
])

lasso_params = {"model__alpha": np.logspace(-4, 1, 50)}

lasso_grid = GridSearchCV(
    lasso_pipeline,
    lasso_params,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

lasso_grid.fit(X_train, y_train)

best_lasso = lasso_grid.best_estimator_
lasso_train_pred = best_lasso.predict(X_train)
lasso_test_pred = best_lasso.predict(X_test)

results.append(
    evaluate_model("Lasso Regression", y_train, lasso_train_pred, y_test, lasso_test_pred)
)

print("Best Lasso Parameter:")
print(lasso_grid.best_params_)
print("Lasso Regression completed successfully!")

## 8. Performance & Coefficient Comparison

**Aim:** Compare the three models side by side.

**Objective:** Tabulate error metrics and learned coefficients for Linear, Ridge and Lasso.

**Theory:** Comparing train vs test metrics reveals overfitting. Comparing coefficients (on standardised features) shows how regularisation changes feature weights and which features matter most.

In [ ]:
results_df = pd.DataFrame(results)

print("Performance Comparison:")
print(results_df.round(4).to_string(index=False))

coefficient_df = pd.DataFrame({
    "Feature": X.columns,
    "Linear": linear_model.named_steps["model"].coef_,
    "Ridge": best_ridge.named_steps["model"].coef_,
    "Lasso": best_lasso.named_steps["model"].coef_
})

print("\nCoefficient Comparison:")
print(coefficient_df.round(4).to_string(index=False))

## 9. Feature Selection using Lasso

**Aim:** Identify which features Lasso keeps or removes.

**Objective:** Inspect Lasso coefficients and list features with (near) zero weight.

**Theory:** Features whose Lasso coefficient is driven to zero contribute nothing to the prediction and are effectively removed. If none are removed, the chosen `alpha` is small and all features are considered useful.

In [ ]:
lasso_coef = pd.Series(best_lasso.named_steps["model"].coef_, index=X.columns)

print("Lasso Coefficients:")
print(lasso_coef.round(4))

tolerance = 1e-8
selected_features = lasso_coef[np.abs(lasso_coef) > tolerance].index.tolist()
removed_features = lasso_coef[np.abs(lasso_coef) <= tolerance].index.tolist()

print("\nFeatures Selected by Lasso:")
print(selected_features)

print("\nFeatures Removed by Lasso:")
print(removed_features)

## 10. Visualisation – Train vs Test R²

**Aim:** Visualise model fit on training and testing data.

**Objective:** Plot Train R² and Test R² for each model.

**Theory:** A large gap between train and test R² indicates overfitting; similar values indicate good generalisation.

In [ ]:
results_df.set_index("Model")[["Train R2", "Test R2"]].plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Training vs Testing R²")
plt.ylabel("R² Score")
plt.xlabel("Regression Model")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Visualisation – Coefficient Comparison

**Aim:** Visualise how coefficients differ across models.

**Objective:** Plot Linear, Ridge and Lasso coefficients for every feature.

**Theory:** Because features are standardised, coefficient magnitudes are directly comparable. Larger absolute values mean stronger influence on the predicted house value.

In [ ]:
coefficient_df.set_index("Feature").plot(
    kind="bar",
    figsize=(13, 6)
)

plt.title("Linear vs Ridge vs Lasso Coefficients")
plt.ylabel("Coefficient (standardized features)")
plt.xlabel("California Housing Feature")
plt.xticks(rotation=45, ha="right")
plt.axhline(y=0, linestyle="--")
plt.tight_layout()
plt.show()

## 12. Best Model Selection

**Aim:** Select the best-performing model.

**Objective:** Choose the model with the lowest Test MSE and report its metrics.

**Theory:** Test-set error estimates performance on unseen data, so the model with the lowest test MSE generalises best.

In [ ]:
best_model_row = results_df.loc[results_df["Test MSE"].idxmin()]

print("Best Model Based on Test MSE:")
print(best_model_row["Model"])
print("Test MSE:", round(best_model_row["Test MSE"], 4))
print("Test RMSE:", round(best_model_row["Test RMSE"], 4))
print("Test MAE:", round(best_model_row["Test MAE"], 4))
print("Test R2:", round(best_model_row["Test R2"], 4))

## Conclusion

Linear, Ridge and Lasso performed almost identically (Test R² ≈ 0.576). Lasso gave the lowest Test MSE (0.5551) by a very small margin and removed no features. `MedInc` is the strongest positive predictor, while `Latitude` and `Longitude` have strong negative coefficients. Moderate VIF values (up to ~9.2 for Latitude) show some multicollinearity, but it is not severe enough to make regularisation matter much here.